In [ ]:
# ==================== 自動處理「TRAIN 0-5」資料夾 ====================

from pathlib import Path
import pandas as pd

# 1. 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. 設定您的資料夾路徑（請確認路徑正確）
TRAIN_FOLDER = "/content/drive/MyDrive//Time-LLM-main/TRAIN 0-5"   # ← 如果資料夾名稱不同，請修改這裡

# 3. 自動掃描資料夾內所有 CSV 並合併成 train.csv
csv_files = list(Path(TRAIN_FOLDER).glob("*.csv"))

print(f"找到 {len(csv_files)} 個 CSV 檔案（資料夾：TRAIN 0-5）")

df_list = []
for file in csv_files:
    df = pd.read_csv(file)
    # 只保留需要的兩欄（您可以自行增加其他欄位）
    if 'disp_x_diff' in df.columns and 'disp_z_diff' in df.columns:
        df = df[['disp_x_diff', 'disp_z_diff']]
        df_list.append(df)
    else:
        print(f"警告：檔案 {file.name} 缺少 disp_x_diff 或 disp_z_diff 欄位，已跳過")

# 合併並儲存
train_df = pd.concat(df_list, ignore_index=True)
train_df.to_csv("/content/drive/MyDrive/train.csv", index=False)

print(f"✅ 合併完成！共 {len(train_df)} 筆資料，已儲存為 train.csv")
print(f"train.csv 路徑：/content/drive/MyDrive/train.csv")

In [ ]:
from huggingface_hub import login

# 執行這段後，下方會出現一個對話框，請貼上你的 Hugging Face Access Token
login()

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate bitsandbytes

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from google.colab import drive
# ==========================================
# 1. 重建 ACE 框架
# ==========================================
class ACE_Playbook:
    def __init__(self):
        self.playbook = []
    def generator(self, trend, top5_lags, expert_knowledge):
        return f"[ACE 動態上下文]\n趨勢：{trend}\n前五大滯後：{top5_lags}\n專家知識：{expert_knowledge}\n建議：溫度與壓力對位移影響大。"
    def curator(self, new_strategy):
        self.playbook.append(new_strategy)
        return self.playbook[-3:]

# ==========================================
# 2. 重建 資料集類別 (解決您的 NameError)
# ==========================================
class LatheDataset(Dataset):
    def __init__(self, csv_path, train_mean=None, train_std=None):
        self.df = pd.read_csv(csv_path).dropna()
        raw_data = torch.tensor(self.df[['disp_x_diff', 'disp_z_diff']].values, dtype=torch.float32)

        if train_mean is None or train_std is None:
            self.mean = raw_data.mean(dim=0)
            self.std = raw_data.std(dim=0)
        else:
            self.mean = train_mean
            self.std = train_std

        self.data = (raw_data - self.mean) / (self.std + 1e-8)
        self.seq_len = 16

    def __len__(self):
        return len(self.data) - self.seq_len + 1

    def __getitem__(self, idx):
        return self.data[idx : idx + self.seq_len]

# ==========================================
# 3. 重建 核心模型類別
# ==========================================
class TimeLLMWithACE(nn.Module):
    def __init__(self, V_prime=200):
        super().__init__()
        bnb_config = BitsAndBytesConfig(load_in_8bit=True)
        self.llm = AutoModelForCausalLM.from_pretrained(
            "meta-llama/Llama-2-7b-hf",
            dtype=torch.float16,
            device_map="auto",
            quantization_config=bnb_config
        )
        self.tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
        self.llm.eval()
        for param in self.llm.parameters():
            param.requires_grad = False

        self.patch_embedding = nn.Linear(2, V_prime)
        self.W = nn.Parameter(torch.empty(V_prime, self.llm.config.vocab_size))
        nn.init.xavier_uniform_(self.W)
        self.reprogram_norm = nn.LayerNorm(self.llm.config.hidden_size)
        self.output_projection = nn.Linear(self.llm.config.hidden_size, 2)
        self.ace = ACE_Playbook()

    def forward(self, patches, trend, top5_lags, expert_knowledge):
        new_strategy = self.ace.generator(trend, top5_lags, expert_knowledge)
        prompt = f"[ACE 動態上下文]\n{new_strategy}\n請預測下一個時間步的位移變化。"
        batch_size = patches.shape[0]

        patch_emb = self.patch_embedding(patches)
        llm_weight_fp32 = self.llm.get_input_embeddings().weight.to(torch.float32)
        E_prime = torch.matmul(self.W, llm_weight_fp32)
        reprogrammed = torch.matmul(patch_emb, E_prime)
        reprogrammed = self.reprogram_norm(reprogrammed).to(torch.float16)

        inputs = self.tokenizer(prompt, return_tensors="pt").to(patch_emb.device)
        text_emb = self.llm.get_input_embeddings()(inputs.input_ids)
        text_emb = text_emb.expand(batch_size, -1, -1)
        inputs_embeds = torch.cat([reprogrammed, text_emb], dim=1)

        outputs = self.llm(inputs_embeds=inputs_embeds, output_hidden_states=True)
        hidden = outputs.hidden_states[-1].mean(dim=1)
        return self.output_projection(hidden.to(torch.float32))

print("✅ 所有地基類別（Dataset, Model, ACE）皆已重新載入記憶體！")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import re
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm.notebook import tqdm

# ==========================================
# 1. 資料對齊與合併模組
# ==========================================
def prepare_data(env_path, source_folder, save_path):
    print(f"開始處理資料夾：{source_folder}")

    env_df = pd.read_excel(env_path)
    new_columns = ['日期', '段1_轉速', '段1_進給', '段1_時間', '段2_轉速', '段2_進給', '段2_時間', '段3_轉速', '段3_進給', '段3_時間', '控溫模式', '溫度']
    env_df.columns = new_columns
    env_df = env_df.drop(0).reset_index(drop=True)

    env_dict = {}
    for _, row in env_df.iterrows():
        if pd.isna(row['日期']): continue

        date_str = str(int(row['日期']))
        temp_mode = str(row['控溫模式'])
        temp_val = str(row['溫度'])
        prompt = f"機台環境為{temp_mode}，設定溫度 {temp_val} 度。"

        stages = []
        if pd.notna(row['段1_轉速']): stages.append(f"第一段轉速 {int(row['段1_轉速'])}rpm，進給 {int(row['段1_進給'])}")
        if pd.notna(row['段2_轉速']): stages.append(f"第二段轉速 {int(row['段2_轉速'])}rpm，進給 {int(row['段2_進給'])}")
        if stages:
            prompt += " 加工參數：" + "；".join(stages) + "。"

        env_dict[date_str] = prompt

    csv_files = list(Path(source_folder).glob("*.csv"))
    print(f"掃描完畢，共找到 {len(csv_files)} 個 CSV 檔案。")

    df_list = []
    for file in csv_files:
        match = re.search(r'2020\d{4}', file.name)
        if not match:
            continue

        date_key = match.group(0)
        matched_prompt = env_dict.get(date_key, "機台運作中，無特殊環境紀錄。")

        df = pd.read_csv(file)
        if 'disp_x_diff' in df.columns and 'disp_z_diff' in df.columns:
            df = df[['disp_x_diff', 'disp_z_diff']].copy()

            # 加入資料清洗邏輯：強制轉數值並剔除缺失值
            df['disp_x_diff'] = pd.to_numeric(df['disp_x_diff'], errors='coerce')
            df['disp_z_diff'] = pd.to_numeric(df['disp_z_diff'], errors='coerce')
            df = df.dropna()

            # 確認清洗後資料不為空，才寫入提示詞並加入合併清單
            if not df.empty:
                df['expert_prompt'] = matched_prompt
                df_list.append(df)

    if len(df_list) > 0:
        final_df = pd.concat(df_list, ignore_index=True)
        final_df.to_csv(save_path, index=False)
        print(f"資料處理完成，共提取 {len(final_df)} 筆有效資料，已儲存至：{save_path}")
        return save_path
    else:
        print(f"錯誤：在 {source_folder} 中未提取到有效資料。")
        return None

# ==========================================
# 2. ACE 框架與核心模型
# ==========================================
class ACE_Playbook:
    def __init__(self):
        self.playbook = []

    def generator(self, trend, top5_lags, expert_knowledge):
        return f"[ACE 動態上下文]\n趨勢：{trend}\n前五大滯後：{top5_lags}\n專家知識：{expert_knowledge}\n建議：溫度與壓力對位移影響大。"

    def curator(self, new_strategy):
        self.playbook.append(new_strategy)
        return self.playbook[-3:]

class TimeLLMWithACE(nn.Module):
    def __init__(self, V_prime=200):
        super().__init__()
        bnb_config = BitsAndBytesConfig(load_in_8bit=True)
        self.llm = AutoModelForCausalLM.from_pretrained(
            "meta-llama/Llama-2-7b-hf",
            dtype=torch.float16,
            device_map="auto",
            quantization_config=bnb_config
        )
        self.tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
        self.llm.eval()

        for param in self.llm.parameters():
            param.requires_grad = False

        self.patch_embedding = nn.Linear(2, V_prime)
        self.W = nn.Parameter(torch.empty(V_prime, self.llm.config.vocab_size))
        nn.init.xavier_uniform_(self.W)
        self.reprogram_norm = nn.LayerNorm(self.llm.config.hidden_size)
        self.output_projection = nn.Linear(self.llm.config.hidden_size, 2)
        self.ace = ACE_Playbook()

    def forward(self, patches, trend, top5_lags, expert_knowledge):
        new_strategy = self.ace.generator(trend, top5_lags, expert_knowledge)
        prompt = f"[ACE 動態上下文]\n{new_strategy}\n請預測下一個時間步的位移變化。"
        batch_size = patches.shape[0]

        patch_emb = self.patch_embedding(patches)
        llm_weight_fp32 = self.llm.get_input_embeddings().weight.to(torch.float32)
        E_prime = torch.matmul(self.W, llm_weight_fp32)
        reprogrammed = torch.matmul(patch_emb, E_prime)
        reprogrammed = self.reprogram_norm(reprogrammed).to(torch.float16)

        inputs = self.tokenizer(prompt, return_tensors="pt").to(patch_emb.device)
        text_emb = self.llm.get_input_embeddings()(inputs.input_ids)
        text_emb = text_emb.expand(batch_size, -1, -1)

        inputs_embeds = torch.cat([reprogrammed, text_emb], dim=1)
        outputs = self.llm(inputs_embeds=inputs_embeds, output_hidden_states=True)

        hidden = outputs.hidden_states[-1].mean(dim=1)
        return self.output_projection(hidden.to(torch.float32))

# ==========================================
# 3. 資料集類別
# ==========================================
class LatheDatasetWithPrompt(Dataset):
    def __init__(self, csv_path, train_mean=None, train_std=None):
        self.df = pd.read_csv(csv_path).dropna()
        self.raw_data = torch.tensor(self.df[['disp_x_diff', 'disp_z_diff']].values, dtype=torch.float32)

        if train_mean is None or train_std is None:
            self.mean = self.raw_data.mean(dim=0)
            self.std = self.raw_data.std(dim=0)
        else:
            self.mean = train_mean
            self.std = train_std

        self.data = (self.raw_data - self.mean) / (self.std + 1e-8)
        self.prompts = self.df['expert_prompt'].values.tolist()
        self.seq_len = 16

    def __len__(self):
        return len(self.data) - self.seq_len + 1

    def __getitem__(self, idx):
        seq = self.data[idx : idx + self.seq_len]
        prompt = self.prompts[idx + self.seq_len - 1]
        return seq, prompt

# ==========================================
# 4. 包含驗證機制的訓練迴圈
# ==========================================
def train_model_integrated(model, train_loader, val_loader, epochs=5, lr=1e-4):
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    criterion = nn.MSELoss()

    print(f"開始訓練程序，總計回合數：{epochs}")
    print("-" * 50)

    for epoch in range(epochs):
        # 訓練階段
        model.train()
        total_train_loss = 0
        train_last_context = ""

        train_progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)

        for batch_seqs, batch_prompts in train_progress:
            optimizer.zero_grad()

            inputs = batch_seqs.cuda()
            target = inputs[:, -1, :].cuda()
            current_env_text = batch_prompts[0]

            trend = "upward" if inputs.mean() > 0 else "downward"
            abs_x_fluctuations = torch.abs(inputs[0, :, 0])
            _, top_indices = torch.topk(abs_x_fluctuations, k=5)
            top5_lags = top_indices.tolist()

            pred = model(inputs, trend, top5_lags, expert_knowledge=current_env_text)
            train_last_context = model.ace.generator(trend, top5_lags, current_env_text)

            loss = criterion(pred, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            total_train_loss += loss.item()
            train_progress.set_postfix({'loss': f"{loss.item():.4f}"})

        avg_train_loss = total_train_loss / len(train_loader)

        # 擷取訓練集最後一筆資料
        train_input_x = inputs[0, :, 0].cpu().detach().numpy().tolist()
        train_pred = pred[0].cpu().detach().numpy().tolist()
        train_actual = target[0].cpu().detach().numpy().tolist()

        # 驗證階段
        model.eval()
        total_val_loss = 0
        val_last_context = ""
        val_progress = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False)

        with torch.no_grad():
            for batch_seqs, batch_prompts in val_progress:
                inputs = batch_seqs.cuda()
                target = inputs[:, -1, :].cuda()
                current_env_text = batch_prompts[0]

                trend = "upward" if inputs.mean() > 0 else "downward"
                abs_x_fluctuations = torch.abs(inputs[0, :, 0])
                _, top_indices = torch.topk(abs_x_fluctuations, k=5)
                top5_lags = top_indices.tolist()

                pred = model(inputs, trend, top5_lags, expert_knowledge=current_env_text)
                val_last_context = model.ace.generator(trend, top5_lags, current_env_text)

                loss = criterion(pred, target)
                total_val_loss += loss.item()

        avg_val_loss = total_val_loss / len(val_loader)

        # 擷取驗證集最後一筆資料
        val_input_x = inputs[0, :, 0].cpu().detach().numpy().tolist()
        val_pred = pred[0].cpu().detach().numpy().tolist()
        val_actual = target[0].cpu().detach().numpy().tolist()

        print(f"\n[Epoch {epoch+1}/{epochs}] 結算報告：")
        print(f"Training Loss   : {avg_train_loss:.5f}")
        print(f"Validation Loss : {avg_val_loss:.5f}")
        print("-" * 50)
        print("【模型輸入與輸出資訊檢視 - 訓練集】")
        print("[訓練集輸入特徵 1] 過去 16 步 X 軸位移正規化數值：")
        print([round(num, 4) for num in train_input_x])
        print(f"\n[訓練集輸入特徵 2] ACE 動態上下文提示詞：\n{train_last_context}")
        print(f"\n[訓練集模型輸出] 預測下一時間步位移差：X 軸 {train_pred[0]:.4f}, Z 軸 {train_pred[1]:.4f}")
        print(f"[訓練集真實答案] 實際下一時間步位移差：X 軸 {train_actual[0]:.4f}, Z 軸 {train_actual[1]:.4f}")
        print("-" * 50)
        print("【模型輸入與輸出資訊檢視 - 驗證集】")
        print("[驗證集輸入特徵 1] 過去 16 步 X 軸位移正規化數值：")
        print([round(num, 4) for num in val_input_x])
        print(f"\n[驗證集輸入特徵 2] ACE 動態上下文提示詞：\n{val_last_context}")
        print(f"\n[驗證集模型輸出] 預測下一時間步位移差：X 軸 {val_pred[0]:.4f}, Z 軸 {val_pred[1]:.4f}")
        print(f"[驗證集真實答案] 實際下一時間步位移差：X 軸 {val_actual[0]:.4f}, Z 軸 {val_actual[1]:.4f}")
        print("=" * 50)

    save_path = "/content/drive/MyDrive/time_llm_ace_env_v2.pth"
    torch.save(model.state_dict(), save_path)
    print(f"訓練程序結束，模型權重已儲存至：{save_path}")

# ==========================================
# 5. 主程式執行區塊
# ==========================================
if __name__ == "__main__":
    print("系統初始化中...")

    ENV_EXCEL_PATH = "/content/drive/MyDrive/Time-LLM-main/檔案環境設定總表.xlsx"
    TRAIN_FOLDER_PATH = "/content/drive/MyDrive/Time-LLM-main/TRAIN 0-5"
    TEST_FOLDER_PATH = "/content/drive/MyDrive/Time-LLM-main/初賽測驗用數據"

    TRAIN_CSV_PATH = "/content/drive/MyDrive/train_env.csv"
    TEST_CSV_PATH = "/content/drive/MyDrive/test_env.csv"

    try:
        print("\n[階段一] 處理訓練資料")
        prepare_data(ENV_EXCEL_PATH, TRAIN_FOLDER_PATH, TRAIN_CSV_PATH)

        print("\n[階段二] 處理驗證資料")
        prepare_data(ENV_EXCEL_PATH, TEST_FOLDER_PATH, TEST_CSV_PATH)

        print("\n[階段三] 載入資料集與模型")
        # 將 batch_size 提升為 8 以加速訓練
        train_dataset = LatheDatasetWithPrompt(TRAIN_CSV_PATH)
        train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

        val_dataset = LatheDatasetWithPrompt(
            TEST_CSV_PATH,
            train_mean=train_dataset.mean,
            train_std=train_dataset.std
        )
        val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

        model = TimeLLMWithACE().cuda()

        print("\n[階段四] 啟動訓練")
        # 設定為 5 個回合
        train_model_integrated(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=5
        )

    except Exception as e:
        print(f"執行期間發生錯誤：{e}")